# 03 — Topic Modeling (YouTube-per-video aggregation)

In [ ]:
from pathlib import Path
import os

BASE = Path("..")
DATA_DIR = BASE / "data"
RAW = DATA_DIR / "raw"
PROC = DATA_DIR / "processed"
FIG = BASE / "figures"

for p in (RAW, PROC, FIG):
    os.makedirs(p, exist_ok=True)

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
import numpy as np

# prefer tokens.parquet
tokens_path = PROC / "clean_tokens.parquet"
if tokens_path.exists():
    df = pd.read_parquet(tokens_path)
else:
    raise FileNotFoundError(f"Missing {tokens_path}. Please re-run Notebook 2.")

# load 'tokens_nostop' column, which is already a list of clean tokens
df_yt = df[df["source"].eq("youtube")].copy()
df_rp = df[df["source"].eq("rappler")].copy()

# join the token lists into a single string for vectorization
df_yt['text'] = df_yt['tokens_nostop'].apply(lambda tokens: ' '.join(tokens))
df_rp['text'] = df_rp['tokens_nostop'].apply(lambda tokens: ' '.join(tokens))

# rename column named "source" to 'platform'.
yt_docs = df_yt[["text", "source"]].rename(columns={"source": "platform"})
rp_docs = df_rp[["text", "source"]].rename(columns={"source": "platform"})

docs = pd.concat([yt_docs, rp_docs], ignore_index=True)
docs = docs.dropna(subset=["text"]).query("text.str.len()>0")

print("Total documents for modeling:", len(docs))

In [ ]:
# loaded stopwords here to ensure they are removed.

def read_lines(path):
    p = BASE / path
    if not p.exists():
        p = Path(path)
    
    if not p.exists():
        print(f"Warning: Stopword file not found: {path}")
        return []
        
    with open(p, "r", encoding="utf-8", errors="ignore") as f:
        # skips comments
        return [ln.strip().lower() for ln in f if ln.strip() and not ln.strip().startswith("#")]

# use the correct filenames from Notebook 2
basic_sw = set(read_lines("configs/Basic Stopwords.txt"))
domain_sw = set(read_lines("configs/Domain-Specific Stopwords.txt"))
tagalog_sw = set(read_lines("configs/tagalog_stopwords.txt"))

STOPWORDS = list(basic_sw | domain_sw | tagalog_sw)
print(f"Loaded {len(STOPWORDS)} stopwords.")

In [ ]:
# Vectorize + LDA

cv = CountVectorizer(
    stop_words=STOPWORDS,
    ngram_range=(1, 2),  # looks for 1-word and 2-word phrases
    min_df=10,           # must appear in at least 10 documents
    max_df=0.85          # ignore words that are in > 85% of all documents
)
X = cv.fit_transform(docs["text"].tolist())

print(f"Matrix shape: {X.shape}")

# rest of the cell is the same
lda = LatentDirichletAllocation(
    n_components=7, learning_method="batch",
    random_state=42, evaluate_every=5
)
lda.fit(X)

def top_terms(model, feat, n=12):
    comps = model.components_
    for t, row in enumerate(comps):
        terms = np.array(feat)[row.argsort()[-n:][::-1]]
        print(f"Topic {t}: {', '.join(terms)}")

top_terms(lda, cv.get_feature_names_out())

In [ ]:
# Top terms + bar charts
import numpy as np, matplotlib.pyplot as plt, os

vocab = cv.get_feature_names_out()

def top_terms(model, vocab, n=12):
    tops=[]
    for t,row in enumerate(model.components_):
        idx=row.argsort()[-n:][::-1]
        terms=[vocab[i] for i in idx]
        w=row[idx]
        tops.append((t,terms,w))
    return tops

tops = top_terms(lda, vocab, 12)

(PROC / "topic_top_terms.txt").open("w",encoding="utf-8").write(
    "\n".join([f"Topic {t}: "+", ".join(terms) for t,terms,_ in tops])
)

for t,terms,w in tops:
    plt.figure(figsize=(8,4))
    colors = plt.cm.viridis(np.linspace(0.1, 0.9, len(terms)))
    plt.barh(terms[::-1], w[::-1], color=colors)
    plt.xlabel("weight"); plt.title(f"Topic {t} – top terms"); plt.tight_layout()
    plt.savefig(FIG / f"topic_terms_T{t}.png", dpi=160); plt.close()

print(f"Saved {len(tops)} topic term charts and topic_top_terms.txt")

In [ ]:
# save Aggregated Matrices
topic_term = pd.DataFrame(lda.components_, columns=cv.get_feature_names_out())
doc_topic = pd.DataFrame(lda.transform(X))

topic_term.to_csv(PROC / "topic_term_matrix_agg.csv", index=False)
doc_topic.to_csv(PROC / "doc_topic_matrix_agg.csv", index=False)

# save the doc_id mapping
docs.reset_index().rename(columns={'index':'doc_id'}).to_csv(PROC / "docs_agg_map.csv", index=False)
print("Saved topic/doc matrices (aggregated).")

In [ ]:
import pandas as pd
import numpy as np

# helper function
def top_terms_df(model, vocab, n=15):
    rows = []
    for t, row in enumerate(model.components_):
        idx = row.argsort()[-n:][::-1]
        terms = [vocab[i] for i in idx]
        weights = [row[i] for i in idx]
        for rank, (term, w) in enumerate(zip(terms, weights), start=1):
            rows.append({"topic": t, "rank": rank, "term": term, "weight": w})
    return pd.DataFrame(rows)

# this next set of code is for the Aggregated notebook
df_terms_agg = top_terms_df(lda, cv.get_feature_names_out(), n=15)

# Use the 'PROC' variable defined in the first cell
df_terms_agg.to_csv(PROC / "topic_top_terms_agg.csv", index=False)

print("Saved: topic_top_terms_agg.csv")
print(df_terms_agg.head())

In [ ]:
import pyLDAvis
import warnings
import numpy as np

# suppress a common warning from the library
warnings.filterwarnings("ignore", category=DeprecationWarning) 

print("Preparing pyLDAvis visualization...")
pyLDAvis.enable_notebook()

# 1. Topic-Term Distributions
topic_term_dists = lda.components_

# 2. Document-Topic Distributions
doc_topic_dists = lda.transform(X)

# 3. Document Lengths
doc_lengths = np.asarray(X.sum(axis=1)).ravel()

# 4. Vocab
vocab = cv.get_feature_names_out()

# 5. Term Frequency
term_frequency = np.asarray(X.sum(axis=0)).ravel()

# pass all 5 arguments to the prepare function
vis_data = pyLDAvis.prepare(
    topic_term_dists=topic_term_dists,
    doc_topic_dists=doc_topic_dists,
    doc_lengths=doc_lengths,
    vocab=vocab,
    term_frequency=term_frequency
)

# save it as an HTML file
vis_path = PROC / 'lda_visualization_agg.html'
pyLDAvis.save_html(vis_data, str(vis_path))

print(f"Saved interactive LDA viz to: {vis_path}")

# display it directly in the notebook
vis_data

In [ ]:
# !pip install wordcloud
from wordcloud import WordCloud
import matplotlib.pyplot as plt

print("\n--- Generating Word Clouds per Topic ---")

vocab = cv.get_feature_names_out()
topic_term_matrix = lda.components_

# loop through each topic
for i, topic_weights in enumerate(topic_term_matrix):

    # this block of code is for creating a dictionary of {word: weight} for this topic's top 50 words
    topic_freqs = {vocab[j]: topic_weights[j] for j in topic_weights.argsort()[-50:][::-1]}

    wc = WordCloud(width=800, 
                   height=400, 
                   background_color='white',
                   colormap='viridis').generate_from_frequencies(topic_freqs)

    plt.figure(figsize=(10, 5))
    plt.imshow(wc, interpolation='bilinear')
    plt.axis('off')
    plt.title(f'Word Cloud for Topic {i}')
    plt.tight_layout()

    # save the figure
    plt.savefig(FIG / f'word_cloud_topic_{i}.png', dpi=160)
    plt.close()

print(f"Saved {len(topic_term_matrix)} word clouds to your 'figures' folder.")